# Limpieza de distritos.csv

Tabla maestra de los 21 distritos de Madrid: id, nombre, código postal principal, población, superficie, número de hospitales y si tiene base del SAMUR.

El CSV trae 24 filas porque hay distritos repetidos con otra escritura. Al final guardamos `distritos_limpio.csv` con 21 filas, una por distrito.

**Orden:** este notebook se ejecuta después de `limpieza_socioeconomico.ipynb`, porque rellena huecos cruzando con `socioeconomico_limpio.csv`.

El CSV original no se toca: el resultado se guarda aparte en `distritos_limpio.csv`.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# El CSV original debe estar junto al notebook o en la carpeta data del repo.
ruta = Path('distritos.csv')
if not ruta.exists():
    ruta = Path('../data/distritos.csv')
dis = pd.read_csv(ruta)
dis.head()

,distrito_id,nombre,codigo_postal_principal,poblacion,superficie_km2,num_hospitales,base_samur
0,1,CENTRO,28012,NaN,5.23,3,1
1,2,ARGANZUELA,28005,155115.0,6.47,2,1
2,3,RETIRO,28007,119915.0,NaN,1,0
3,4,SALAMANCA,28001,147707.0,NaN,2,0
4,5,CHAMARTIN,28002,146512.0,9.19,3,1


## Revisión inicial

Miramos los datos antes de tocar nada: cuántas filas hay, dónde están los huecos y qué valores raros aparecen.

In [2]:
print(dis.shape)          # 24 filas, pero hay 21 distritos
print()
print(dis.isna().sum())   # huecos en poblacion y superficie
print()
print(f"Poblacion minima: {dis['poblacion'].min()}")   # -999: valor centinela
print()
print(dis[['distrito_id', 'nombre']].to_string())

(24, 7)

distrito_id                0
nombre                     0
codigo_postal_principal    0
poblacion                  3
superficie_km2             2
num_hospitales             0
base_samur                 0
dtype: int64

Poblacion minima: -999.0

    distrito_id               nombre
0             1               CENTRO
1             2           ARGANZUELA
2             3               RETIRO
3             4            SALAMANCA
4             5            CHAMARTIN
5             6               TETUAN
6             7             CHAMBERI
7             8  FUENCARRAL-EL PARDO
8             9      MONCLOA-ARAVACA
9            10               LATINA
10           11          CARABANCHEL
11           12                USERA
12           13   PUENTE DE VALLECAS
13           14            MORATALAZ
14           15        CIUDAD LINEAL
15           16            HORTALEZA
16           17           VILLAVERDE
17           18    VILLA DE VALLECAS
18           19            VICALVARO
19      

## Nombres de distrito

Hay distritos repetidos escritos de otra forma (`Centro`, `Chamartin`). Usamos la misma regla que en el socioeconómico: mayúsculas, sin tildes y sin espacios sobrantes; si el texto es el principio de un único nombre oficial, se completa; y si no se reconoce, se deja y se lista para revisarlo. Nada de diccionarios cerrados, como nos pidió Bluetab.

In [3]:
OFICIALES = ["CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
             "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
             "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA", "VILLAVERDE",
             "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"]

def normalizar_distrito(nombre):
    if pd.isna(nombre):
        return None
    s = unicodedata.normalize('NFD', str(nombre).strip().upper())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = ' '.join(s.split())
    if s in OFICIALES:
        return s
    candidatos = [o for o in OFICIALES if o.startswith(s)]
    if len(candidatos) == 1:
        return candidatos[0]
    return None

dis['nombre'] = dis['nombre'].map(normalizar_distrito)
sin_asignar = dis[dis['nombre'].isna()]
print(f"Filas sin distrito reconocido: {len(sin_asignar)}")
if len(sin_asignar):
    print(sin_asignar)

Filas sin distrito reconocido: 0


## Distritos repetidos

Tres distritos salen dos veces: Centro (ids 1 y 22), Tetuán (6 y 23) y Chamartín (5 y 24).

- Las copias de Tetuán y Chamartín son idénticas.
- La primera copia de Centro no tiene población y la segunda sí: juntando las dos, Centro se queda completo.

Nos quedamos una fila por distrito con su id original (del 1 al 21), rellenando huecos con la otra copia. Antes comprobamos que ninguna pareja se contradiga.

In [4]:
cols_num = ['codigo_postal_principal', 'poblacion', 'superficie_km2', 'num_hospitales', 'base_samur']

# Distritos con dos valores distintos (no nulos) en alguna columna
conflictos = dis.groupby('nombre')[cols_num].nunique() > 1
cols_conflicto = conflictos.columns[conflictos.any()].tolist()
if not cols_conflicto:
    print("Sin contradicciones entre copias: solo se completan huecos.")
for c in cols_conflicto:
    for d in conflictos.index[conflictos[c]]:
        print(f"Conflicto en {d} / {c}: {dis.loc[dis['nombre']==d, c].tolist()}")

dis = dis.sort_values('distrito_id').groupby('nombre', as_index=False).first()
dis['distrito_id'] = dis['distrito_id'].astype(int)
print()
print(dis.shape)   # (21, 7)
print(f"IDs: {dis['distrito_id'].tolist()}")

Sin contradicciones entre copias: solo se completan huecos.

(21, 7)
IDs: [2, 21, 11, 1, 5, 7, 15, 8, 16, 10, 9, 14, 13, 3, 4, 20, 6, 12, 19, 18, 17]


## Valores imposibles

Moratalaz tiene población -999: es un valor centinela, no una población real. Lo pasamos a vacío (luego lo recuperamos cruzando con el socioeconómico). De paso comprobamos que el resto de columnas tienen valores válidos: códigos postales 280XX, superficie positiva, hospitales ≥ 0 y base_samur 0 o 1.

In [5]:
dis.loc[dis['poblacion'] <= 0, 'poblacion'] = np.nan

cp_malos = dis[~dis['codigo_postal_principal'].astype(str).str.match(r'280\d{2}')]
print(f"Codigos postales que no son 280XX: {len(cp_malos)}")
print(f"Superficies no positivas: {(dis['superficie_km2'] <= 0).sum()}")
print(f"num_hospitales negativos: {(dis['num_hospitales'] < 0).sum()}")
print(f"base_samur distinto de 0/1: {(~dis['base_samur'].isin([0, 1])).sum()}")

Codigos postales que no son 280XX: 0
Superficies no positivas: 0
num_hospitales negativos: 0
base_samur distinto de 0/1: 0


## Huecos, cruzando con el socioeconómico

Faltan la población de Moratalaz, San Blas-Canillejas y Barajas, y la superficie de Retiro y Salamanca.

Como son los mismos distritos, los rellenamos (95.840 para Moratalaz, 162.715 para San Blas-Canillejas y 5,46 y 5,41 km² para Retiro y Salamanca) desde `socioeconomico_limpio.csv` (`poblacion_2024` y `superficie_km2`) en vez de inventarlos. Es el mismo cruce entre tablas que luego usamos en el caso.

En Barajas el socioeconómico tampoco tenía población y allí se estimó como densidad × superficie (47.703). Usamos ese mismo valor aquí para que las dos tablas digan lo mismo. Es una estimación, no un dato real.

In [6]:
soc = pd.read_csv('socioeconomico_limpio.csv')

dis = dis.merge(soc[['distrito', 'poblacion_2024', 'superficie_km2']]
                  .rename(columns={'distrito': 'nombre',
                                   'poblacion_2024': 'poblacion_soc',
                                   'superficie_km2': 'superficie_soc'}),
                on='nombre', how='left')

dis['poblacion'] = dis['poblacion'].fillna(dis['poblacion_soc'])
dis['superficie_km2'] = dis['superficie_km2'].fillna(dis['superficie_soc'])

# Coherencia con el socioeconomico: misma poblacion y superficie por distrito
descuadra = dis[(dis['poblacion'] != dis['poblacion_soc']) |
                ((dis['superficie_km2'] - dis['superficie_soc']).abs() > 0.01)]
print(f"Distritos que no cuadran con el socioeconomico: {len(descuadra)}")

dis = dis.drop(columns=['poblacion_soc', 'superficie_soc'])

print("Huecos que quedan:")
print(dis.isna().sum()[dis.isna().sum() > 0])

Distritos que no cuadran con el socioeconomico: 0
Huecos que quedan:
Series([], dtype: int64)


## Estadísticos y guardado

Dejamos las 21 filas ordenadas por `distrito_id` y miramos los estadísticos finales.

Los datos de `num_hospitales` y `base_samur` los dejamos como vienen. Más adelante, en `cruces_datasets.ipynb`, se comparan con los catálogos `hospitales.csv` y `bases_samur.csv` (y ahí se ve que no coinciden en varios distritos).

In [7]:
dis = dis.sort_values('distrito_id').reset_index(drop=True)
# Mismo orden de columnas que el original
dis = dis[['distrito_id', 'nombre', 'codigo_postal_principal', 'poblacion',
           'superficie_km2', 'num_hospitales', 'base_samur']]
dis['poblacion'] = dis['poblacion'].astype(int)
dis.to_csv('distritos_limpio.csv', index=False)
print(dis.shape)   # (21, 7)
dis.describe().round(2)

(21, 7)


,distrito_id,codigo_postal_principal,poblacion,superficie_km2,num_hospitales,base_samur
count,21.0,21.00,21.00,21.00,21.00,21.00
mean,11.0,28020.62,157604.19,28.80,1.29,0.52
std,6.2,14.36,56809.83,50.08,0.90,0.51
min,1.0,28001.00,47703.00,4.68,0.00,0.00
25%,6.0,28008.00,123606.00,6.31,1.00,0.00
50%,11.0,28019.00,149718.00,14.05,1.00,1.00
75%,16.0,28031.00,193104.00,28.24,2.00,1.00
max,21.0,28052.00,257329.00,237.84,3.00,1.00
